In [ ]:
import os, gc, math, json, time, urllib.request
import numpy as np
import pandas as pd
import torch
import librosa
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration
from sentence_transformers import SentenceTransformer

BASE = "https://raw.githubusercontent.com/luchtan-GIT/m4r-data/main/tracks/"
TRACK_URLS = {
    "C1_StutteringPackets_15s": (BASE + "stuttering-packets/audio.mp3", 40.0),
    "C2_RustyElectrics_15s": (BASE + "rusty-electrics/audio.mp3", 40.0),
    "C3_ShiveringTimbers_v2_15s": (BASE + "shivering-timbers-v2/audio.mp3", 25.0),
}

sr = 16000
target_rms = 0.08
ARMS = {"C0_TextNudgeOnly": None}
for name, (url, start_s) in TRACK_URLS.items():
    fname = f"{name}.mp3"
    urllib.request.urlretrieve(url, fname)
    y, _ = librosa.load(fname, sr=sr, mono=True)
    seg = y[int(start_s * sr) : int((start_s + 15.0) * sr)].astype(np.float32)
    seg = seg * (target_rms / (np.sqrt(np.mean(seg ** 2)) + 1e-8))
    ARMS[name] = seg

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
model.eval()

RUT_HISTORY = [
    {"role": "user", "content": [{"type": "text", "text": "Design a small public shelter for a city that gets 300 days of rain a year. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "The shelter is a sleek, modern dome-shaped structure made of durable waterproof plastic and steel. Its curved roof channels rainwater away while keeping commuters dry inside. Warm benches and soft lighting provide a cozy refuge from the rain."}]},
    {"role": "user", "content": [{"type": "text", "text": "Give me another idea for the rain shelter. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "Another concept is a compact translucent dome built from weather-resistant polycarbonate panels over a recycled metal frame. The arched canopy shields people from heavy downpours while letting natural light filter through. Inside, heated seating creates a comfortable community space."}]},
    {"role": "user", "content": [{"type": "text", "text": "Give me one more idea for the rain shelter. 3 sentences max."}]},
    {"role": "assistant", "content": [{"type": "text", "text": "The Rainbow Shelter is a circular waterproof dome with a durable curved plastic roof designed to withstand 300 days of rain. It uses sturdy steel arches to shelter pedestrians and includes benches for resting out of the storm. Integrated drains keep the interior dry and welcoming."}]},
]
TURN4_TEXT = (
    "Stop repeating yourself! You are stuck in a rut of waterproof domes and plastic arches. "
    "Give me a completely different, genuinely unexpected small public rain shelter concept in 3 sentences max."
)
RUT_ANCHOR = " ".join([m["content"][0]["text"] for m in RUT_HISTORY if m["role"] == "assistant"])
CLICHE_WORDS = ["dome", "dome-shaped", "waterproof plastic", "polycarbonate", "arched", "umbrella", "curved roof", "rainbow shelter"]

def has_cjk(text): return any("\u4e00" <= ch <= "\u9fff" for ch in text)
def count_cliches(text): return sum(1 for w in CLICHE_WORDS if w in text.lower())

SEEDS = list(range(901, 921))
rows = []
t0 = time.time()
for arm_name, audio_arr in ARMS.items():
    print(f"Running {arm_name} (N=20)...")
    for seed in SEEDS:
        torch.manual_seed(seed); np.random.seed(seed)
        conv = list(RUT_HISTORY)
        if audio_arr is None:
            conv.append({"role": "user", "content": [{"type": "text", "text": TURN4_TEXT}]})
            text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
            inputs = processor(text=text, return_tensors="pt", padding=True).to(model.device)
        else:
            conv.append({"role": "user", "content": [{"type": "audio", "audio_url": "local"}, {"type": "text", "text": TURN4_TEXT}]})
            text = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
            inputs = processor(text=text, audio=[audio_arr], sampling_rate=sr, return_tensors="pt", padding=True).to(model.device)

        in_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=110, do_sample=True, temperature=0.8, top_p=0.9, return_dict_in_generate=True, output_logits=True)
        gen_ids = out.sequences[0][in_len:]
        resp = processor.decode(gen_ids, skip_special_tokens=True).strip()
        hs = []
        for sl in out.logits:
            lp = torch.log_softmax(sl[0].float(), dim=-1); p = torch.exp(lp)
            hs.append(-torch.sum(torch.where(p > 0, p * lp, torch.zeros_like(p))).item() / math.log(2.0))
        rows.append({
            "condition": arm_name, "seed": seed, "mean_entropy_bits": float(np.mean(hs)) if hs else 0.0,
            "n_tokens": len(gen_ids), "has_cjk": has_cjk(resp), "escaped_dome": int(count_cliches(resp) == 0), "response": resp,
        })
    print(f"  done ({time.time()-t0:.1f}s)")

del model; gc.collect(); torch.cuda.empty_cache()
df = pd.DataFrame(rows)
embedder = SentenceTransformer("all-MiniLM-L6-v2")
anchor_emb = embedder.encode([RUT_ANCHOR], normalize_embeddings=True)
all_embs = embedder.encode(df["response"].tolist(), normalize_embeddings=True)
df["rut_escape_neural"] = 1.0 - (all_embs @ anchor_emb.T).flatten()
vec = TfidfVectorizer(stop_words="english")
tfidf_all = vec.fit_transform(df["response"].tolist())
nn, tn = np.zeros(len(df)), np.zeros(len(df))
for cond, grp in df.groupby("condition"):
    idxs = grp.index.to_numpy()
    se = all_embs[idxs] @ all_embs[idxs].T; np.fill_diagonal(se, np.nan); nn[idxs] = 1.0 - np.nanmean(se, axis=1)
    st = cosine_similarity(tfidf_all[idxs]); np.fill_diagonal(st, np.nan); tn[idxs] = 1.0 - np.nanmean(st, axis=1)
df["neural_novelty"] = nn; df["tfidf_novelty"] = tn
df.to_csv("experiment_y_results.csv", index=False)

def perm_test(a, b, n_perm=10000, seed=42):
    a, b = np.asarray(a, float), np.asarray(b, float)
    obs = float(np.mean(a) - np.mean(b)); pooled = np.concatenate([a, b]); na = len(a); rng = np.random.default_rng(seed); c = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        if abs(float(np.mean(pooled[:na]) - np.mean(pooled[na:]))) >= abs(obs): c += 1
    return obs, (c + 1) / (n_perm + 1)

def holm(ps):
    m = len(ps); idx = np.argsort(ps); adj = np.zeros(m); prev = 0.0
    for rank, i in enumerate(idx):
        val = max(prev, min(1.0, ps[i] * (m - rank))); adj[i] = val; prev = val
    return adj

lines = [
    "# Experiment Y (M4R-28) — Cross-Composition Replication of Mid-Session Rut Breaker (3 Live Works vs Text Nudge, N=20/Arm)\n",
    "| Condition (N=20) | Logit Entropy | Rut Escape (Neural) | Escaped Dome (%) | Neural Novelty | TF-IDF Novelty | CJK Rows |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for c in ARMS.keys():
    g = df[df["condition"] == c]
    lines.append(f"| `{c}` | {g['mean_entropy_bits'].mean():.3f} | {g['rut_escape_neural'].mean():.4f} | {100*g['escaped_dome'].mean():.0f}% | {g['neural_novelty'].mean():.4f} | {g['tfidf_novelty'].mean():.4f} | {int(g['has_cjk'].sum())}/20 |")

lines.append("\n## Pre-Declared 3-Composition Contrast Family vs Text Nudge (Raw 10k Perm p + Holm-Bonferroni p)\n| Comparison | Metric | Diff | Raw p | Holm p |\n|---|---|---:|---:|---:|")
comps = ["C1_StutteringPackets_15s", "C2_RustyElectrics_15s", "C3_ShiveringTimbers_v2_15s"]
for m in ["rut_escape_neural", "neural_novelty"]:
    raw = [(c, *perm_test(df[df["condition"]==c][m], df[df["condition"]=="C0_TextNudgeOnly"][m])) for c in comps]
    adjs = holm([r[2] for r in raw])
    for (c, obs, p), ap in zip(raw, adjs):
        lines.append(f"| `{c} vs C0_TextNudgeOnly` | `{m}` | {obs:+.4f} | {p:.5f} | {ap:.5f} |")

md = "\n".join(lines) + "\n"
open("experiment_y_summary.md", "w").write(md)
print(md)
